In [ ]:
import sys
sys.path.insert(0, "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender")

import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

from dataset_generator.single_transactions_dataset import SingleTransactionsDataset
from dataset_generator.feature_extraction import (
    NODE_TYPE_FEATURE_DIMS, SINGLE_CHAIN_LABEL_MAP,
    IN_DEGREE_INDEX, OUT_DEGREE_INDEX, AMOUNTS_INDEX,
)

dataset = SingleTransactionsDataset(root="/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset")
print(f"Dataset size: {len(dataset)} graphs")

/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exporting data from database to CSV files... 
Dataset size: 1138 graphs


In [2]:
import os
import csv
import shutil


def split_single_transactions_dataset(
    source_root: str,
    train_root: str,
    test_root: str,
    test_tx_hashes,
    train_tx_hashes=None,
):
    source_processed = os.path.join(source_root, "processed")
    source_raw = os.path.join(source_root, "raw")

    with open(os.path.join(source_processed, "data_index.csv"), "r") as f:
        rows = list(csv.DictReader(f))

    test_tx_hashes = set(test_tx_hashes)
    missing = test_tx_hashes - {r["tx_hash"] for r in rows}
    if missing:
        raise ValueError(f"{len(missing)} train tx_hashes not found in source dataset: {sorted(missing)[:5]}...")

    test_rows = [r for r in rows if r["tx_hash"] in test_tx_hashes]

    if train_tx_hashes is None:
        train_rows = [r for r in rows if r["tx_hash"] not in test_tx_hashes]
    else:
        train_tx_hashes = set(train_tx_hashes)
        missing = train_tx_hashes - {r["tx_hash"] for r in rows}
        if missing:
            raise ValueError(f"{len(missing)} test tx_hashes not found in source dataset: {sorted(missing)[:5]}...")
        overlap = test_tx_hashes & train_tx_hashes
        if overlap:
            raise ValueError(f"{len(overlap)} tx_hashes appear in both splits: {sorted(overlap)[:5]}...")
        train_rows = [r for r in rows if r["tx_hash"] in train_tx_hashes]

    fieldnames = ["index", "graph_id", "bridge", "tx_hash", "file_name", "offset", "synthetic"]

    for split_root, split_rows in [(train_root, train_rows), (test_root, test_rows)]:
        split_raw = os.path.join(split_root, "raw")
        split_processed = os.path.join(split_root, "processed")
        os.makedirs(split_raw, exist_ok=True)
        os.makedirs(split_processed, exist_ok=True)

        # Copy raw CSVs unchanged so SingleTransactionsDataset.__init__ finds
        # raw_file_names present and doesn't trigger a DB download.
        for raw_file in os.listdir(source_raw):
            shutil.copy2(os.path.join(source_raw, raw_file), os.path.join(split_raw, raw_file))

        # Copy only the .pt files this split actually references (dedup, since
        # each file holds up to 50 graphs and several rows may share one file).
        referenced_files = {r["file_name"] for r in split_rows}
        for file_name in referenced_files:
            shutil.copy2(
                os.path.join(source_processed, file_name),
                os.path.join(split_processed, file_name),
            )

        # metadata.json (node/edge type vocab) is dataset-wide, safe to reuse as-is.
        shutil.copy2(
            os.path.join(source_processed, "metadata.json"),
            os.path.join(split_processed, "metadata.json"),
        )

        # Filtered index; file_name/offset are kept as-is since the .pt files were copied verbatim.
        with open(os.path.join(split_processed, "data_index.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for new_idx, r in enumerate(split_rows):
                writer.writerow({**r, "index": new_idx})

    return len(train_rows), len(test_rows)


In [ ]:
# Full-remove nomad-ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 406 graphs, test: 732 graphs


In [5]:
test_df = SingleTransactionsDataset(root=TEST_ROOT)

bridges = set(entry["bridge"] for entry in test_df.data_index)
print(f"Bridges in test set: {bridges}")

Exporting data from database to CSV files... 
Bridges in test set: {'ronin1'}


In [ ]:
# 5050 testing (with 5050 normal 5050 anomaly)
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train_50label"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_50label"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


Exporting data from database to CSV files... 
train: 773 graphs, test: 365 graphs


In [ ]:
# Anomalies-only-remove nomad-ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train_fullanom"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_fullanom"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]
# Filter test_tx_hashes to include only anomalies (y[0] == 1)
filtered_test_tx_hashes = []
for idx, entry in enumerate(source_ds.data_index):
    if entry["tx_hash"] in test_tx_hashes:
        graph = source_ds[idx]
        if graph.y[0] == 1:  # anomaly label
            filtered_test_tx_hashes.append(entry["tx_hash"])

test_tx_hashes = filtered_test_tx_hashes

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")



Exporting data from database to CSV files... 
train: 969 graphs, test: 169 graphs


In [ ]:
# Full-remove only nomad
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train_fullnomad"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_fullnomad"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 408 graphs, test: 730 graphs


In [ ]:
# Full-remove only ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train_fullronin1"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_fullronin1"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1136 graphs, test: 2 graphs
